# MTL — Colab/Kaggle GPU training

Run cells top to bottom. This trains the shared ResNet50+FPN backbone
jointly on detection (RetinaNet) + semantic segmentation (FCN) +
multi-label classification, on a COCO subset.


## 1. Install dependencies
Colab usually ships a CUDA-matched torch/torchvision already — only reinstall if missing.

In [1]:
import torch
print(torch.__version__, torch.cuda.is_available())

# Uncomment only if the above shows no CUDA:
# !pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121

!pip install pycocotools PyYAML tqdm scikit-learn requests

2.11.0+cpu False


## 2. Get the repo
Either clone from git, or upload a zip of this project via the Colab file browser and unzip it.

In [2]:
!git clone https://github.com/itu-itis23-ucgun22/mtl.git
%cd mtl
!pip install -e .

Cloning into 'mtl'...
remote: Enumerating objects: 51, done.
remote: Counting objects: 100% (51/51), done.
remote: Compressing objects: 100% (43/43), done.
remote: Total 51 (delta 4), reused 51 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (51/51), 25.59 KiB | 1.71 MiB/s, done.
Resolving deltas: 100% (4/4), done.
/content/mtl
Obtaining file:///content/mtl
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for mtl (pyproject.toml) ... done
  Created wheel for mtl: filename=mtl-0.1.0-0.editable-py3-none-any.whl size=1180 sha256=35fafbf8abf5583c0cd6429daeb1200445b7431905251cd70fe7f21e83e64ec1
  Stored in directory: /tmp/pip-ephem-wheel-cache-3fell9q5/wheels/b8/3b/63/cd9bbc919e23d84900281b7352d08ba8b6890722307e161497
Successfully built mtl


## 3. Build the COCO subset
Download full COCO annotations/images first (e.g. via the COCO website or Kaggle's COCO dataset), then filter down to the training subset.

In [3]:
!wget -q http://images.cocodataset.org/annotations/annotations_trainval2017.zip
!unzip -q annotations_trainval2017.zip

!python scripts/prepare_coco_subset.py \
    --ann-file annotations/instances_train2017.json \
    --out data/coco_subset/annotations/instances_train_subset.json \
    --n-images 22500
!python scripts/prepare_coco_subset.py \
    --ann-file annotations/instances_val2017.json \
    --out data/coco_subset/annotations/instances_val_subset.json \
    --n-images 2000

# Download only the subset's images (via each image's coco_url) - far
# cheaper than the full train2017.zip (~18GB) / val2017.zip (~1GB).
!python scripts/download_subset_images.py \
    --ann-file data/coco_subset/annotations/instances_train_subset.json \
    --out-dir data/coco_subset/images/train
!python scripts/download_subset_images.py \
    --ann-file data/coco_subset/annotations/instances_val_subset.json \
    --out-dir data/coco_subset/images/val

loading annotations into memory...
Done (t=15.66s)
creating index...
index created!
Wrote 22500 images to data/coco_subset/annotations/instances_train_subset.json
loading annotations into memory...
Done (t=0.87s)
creating index...
index created!
Wrote 2000 images to data/coco_subset/annotations/instances_val_subset.json
downloading images: 100% 22500/22500 [05:49<00:00, 64.42it/s]
Done. 22500/22500 images present in data/coco_subset/images/train.
downloading images: 100% 2000/2000 [00:27<00:00, 72.83it/s]
Done. 2000/2000 images present in data/coco_subset/images/val.


## 4. Train

In [ ]:
!python scripts/train.py --config configs/train_colab_gpu.yaml

[device] CUDA requested but not available - falling back to CPU.
loading annotations into memory...
Done (t=5.33s)
creating index...
index created!
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100% 97.8M/97.8M [00:00<00:00, 107MB/s]
Config: {'data': {'ann_file': 'data/coco_subset/annotations/instances_train_subset.json', 'img_dir': 'data/coco_subset/images/train', 'val_ann_file': 'data/coco_subset/annotations/instances_val_subset.json', 'val_img_dir': 'data/coco_subset/images/val', 'n_images': 22500, 'img_size': 512, 'num_workers': 2}, 'model': {'backbone_name': 'resnet50', 'pretrained': True, 'trainable_backbone_layers': 3, 'cls_head_tap': 'fpn_p5'}, 'loss': {'det_cls': 1.0, 'det_box': 1.0, 'seg': 1.0, 'cls': 0.5}, 'train': {'device': 'cuda', 'batch_size': 8, 'epochs': 16, 'max_steps': None, 'lr': 0.0001, 'weight_decay': 0.0001, 'amp': True, 'seed': 42, 'log_every': 20, 'checkpoint_dir': 'checkpoin

## 5. Evaluate + visualize a few predictions

In [ ]:
!python scripts/eval.py --config configs/train_colab_gpu.yaml --checkpoint checkpoints/colab_gpu_epoch15.pt

In [ ]:
import torch
import matplotlib.pyplot as plt
from torchvision.utils import draw_bounding_boxes

from mtl.config import load_config
from mtl.datasets.coco_multitask import CocoMultiTaskDataset
from mtl.engine.checkpoint import load_checkpoint
from mtl.models.multitask_model import MultiTaskModel
from mtl.utils.device import resolve_device

cfg = load_config("configs/train_colab_gpu.yaml")
device = resolve_device(cfg.train.device)
dataset = CocoMultiTaskDataset(cfg.data.val_ann_file, cfg.data.val_img_dir, img_size=cfg.data.img_size, train=False)

model = MultiTaskModel(
    det_num_classes=dataset.num_classes,
    seg_num_classes=dataset.num_classes + 1,
    cls_num_labels=dataset.num_classes,
).to(device)
load_checkpoint(model, optimizer=None, path="checkpoints/colab_gpu_epoch15.pt", map_location=str(device))
model.eval()

image, target = dataset[0]
with torch.no_grad():
    out = model(image.unsqueeze(0).to(device))

det = out["detections"][0]
keep = det["scores"] > 0.5
img_uint8 = ((image * 0.5 + 0.5) * 255).clamp(0, 255).byte()
drawn = draw_bounding_boxes(img_uint8, det["boxes"][keep].cpu())
plt.imshow(drawn.permute(1, 2, 0))
plt.title(f"top labels: {out['cls_pred'][0].topk(5).indices.tolist()}")
plt.show()